In [ ]:
from nanover.openmm import OpenMMSimulation
from nanover.mdanalysis.converter import frame_data_to_mdanalysis

simulation = OpenMMSimulation.from_xml_path("../openmm/openmm_files/17-ala.xml")
simulation.load()
universe = frame_data_to_mdanalysis(simulation.make_topology_frame())

In [ ]:
from nanover.app import OmniRunner

imd_runner = OmniRunner.with_basic_server(simulation, port=0, name="MOVEABLE RESTRAINTS")
imd_runner.load(0)
imd = imd_runner.app_server.imd

In [ ]:
from nanover.jupyter import show_runner_controls

show_runner_controls(imd_runner)

In [ ]:
from nanover.imd.imd_state import ParticleInteraction

def do_full_velocity_reset():
    for i, residue in enumerate(universe.residues):
        particles = [int(j) for j in residue.atoms.indices]
        imd.insert_interaction(f"interaction.RESET.{i}", ParticleInteraction((0, 0, 0), scale=0, reset_velocities=True, particles=particles))
    simulation.advance_by_one_step()
    for i, residue in enumerate(universe.residues):
        imd.remove_interaction(f"interaction.RESET.{i}")
    simulation.advance_by_one_step()


def on_interaction_started(*, key: str, interaction: ParticleInteraction):
    if "RESET" in key:
        return
    imd_runner.play()
    # imd_runner.runner.play_step_interval = 1/30


def on_interaction_stopped(*, key: str, interaction: ParticleInteraction):
    if "RESET" in key:
        return
    if not imd.active_interactions:
        imd_runner.pause()
        # imd_runner.runner.play_step_interval = 1
        do_full_velocity_reset()

imd_runner.app_server.imd.interaction_started.add_callback(on_interaction_started)
imd_runner.app_server.imd.interaction_stopped.add_callback(on_interaction_stopped)

In [ ]:
from nanover.websocket import NanoverImdClient

with NanoverImdClient.from_runner(imd_runner) as client:
    with client.root_selection.modify() as selection:
        selection.renderer = "cartoon"
        selection.velocity_reset = True
        selection.interaction_method = "group"